# OPS Junior Business Analyst Portfolio Project  
## Relational Healthcare Data → FHIR R4 JSON → API Validation → Databricks SQL

**Dataset:** Kaggle `prasad22/healthcare-dataset`  
**Purpose:** demonstrate practical experience with FHIR resource modeling, FHIR APIs, Agile user stories, source-to-target mapping, JSON creation and manipulation, SQL querying, Databricks/Spark transformations, Postman API testing, technical documentation, and traceability.

> This project uses synthetic healthcare records. Do not use real personal health information in public notebooks or public FHIR test servers.

## 1. JD-to-deliverable traceability matrix

| Job requirement | Notebook evidence |
|---|---|
| FHIR resource modeling | Patient, Encounter, Condition, Observation, MedicationStatement, Coverage, Claim, Practitioner, Organization |
| FHIR APIs | Transaction Bundle generation, optional metadata request, validation request, API search example |
| Agile using Azure DevOps | Import-friendly backlog CSV with user stories, acceptance criteria, tags, and priorities |
| Translate requirements into user stories | Backlog artifact and acceptance criteria |
| Collaborate with technical teams | Mapping notes and assumptions clarify implementation decisions |
| Databricks and SQL | Spark DataFrame, temp view, `%sql` examples, data-quality queries, JSON export |
| Source-to-target mapping | Mapping DataFrame and CSV artifact |
| Relational → FHIR structures | Field-level mapping and transformation functions |
| JSON creation and manipulation | Resource dictionaries, transaction bundles, NDJSON export |
| Postman API testing | Importable collection with requests and tests |

## 2. Install dependencies

For a local environment, run the next cell once. In Databricks, use `%pip install` in a notebook cell.

In [ ]:
# Uncomment when needed:
# %pip install pandas kagglehub requests fhir.resources pyspark

In [ ]:
from pathlib import Path
import json, re, hashlib, os
from datetime import datetime
from decimal import Decimal
import pandas as pd

PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "generated_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

KAGGLE_HANDLE = "prasad22/healthcare-dataset"
DEMO_FILE = PROJECT_DIR / "healthcare_dataset_demo.csv"
print("Output directory:", OUTPUT_DIR.resolve())

## 3. Download the Kaggle dataset or use the included offline fallback

The notebook attempts to use `kagglehub`. If Kaggle download is unavailable, it loads `healthcare_dataset_demo.csv`. This makes the portfolio notebook reproducible during interviews.

In [ ]:
def locate_csv(folder: Path) -> Path:
    preferred = list(folder.rglob("healthcare_dataset.csv"))
    if preferred:
        return preferred[0]
    csv_files = list(folder.rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found under {folder}")
    return csv_files[0]

def load_healthcare_data() -> tuple[pd.DataFrame, str]:
    try:
        import kagglehub
        dataset_dir = Path(kagglehub.dataset_download(KAGGLE_HANDLE))
        csv_path = locate_csv(dataset_dir)
        return pd.read_csv(csv_path), f"Kaggle: {csv_path}"
    except Exception as exc:
        if not DEMO_FILE.exists():
            raise FileNotFoundError(
                "Kaggle download failed and the fallback file is missing. "
                "Place healthcare_dataset_demo.csv beside the notebook."
            ) from exc
        return pd.read_csv(DEMO_FILE), f"Offline fallback: {DEMO_FILE}"

raw_df, source_used = load_healthcare_data()
print(source_used)
print("Rows:", len(raw_df))
raw_df.head()

## 4. Profile and clean the relational source

The cleaning stage standardizes column names, parses dates, converts billing amounts, detects missing values, and creates deterministic synthetic identifiers for traceability.

In [ ]:
COLUMN_MAP = {
    "Name":"patient_name",
    "Age":"age",
    "Gender":"gender",
    "Blood Type":"blood_type",
    "Medical Condition":"medical_condition",
    "Date of Admission":"admission_date",
    "Doctor":"doctor",
    "Hospital":"hospital",
    "Insurance Provider":"insurance_provider",
    "Billing Amount":"billing_amount",
    "Room Number":"room_number",
    "Admission Type":"admission_type",
    "Discharge Date":"discharge_date",
    "Medication":"medication",
    "Test Results":"test_results",
}

missing_expected = sorted(set(COLUMN_MAP) - set(raw_df.columns))
if missing_expected:
    raise ValueError(f"Expected source columns are missing: {missing_expected}")

df = raw_df.rename(columns=COLUMN_MAP).copy()
for col in ["admission_date","discharge_date"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")
df["billing_amount"] = pd.to_numeric(df["billing_amount"], errors="coerce")
df["age"] = pd.to_numeric(df["age"], errors="coerce").astype("Int64")
df["source_record_id"] = [
    hashlib.sha256(f"{name}|{adm}|{i}".encode()).hexdigest()[:16]
    for i, (name, adm) in enumerate(zip(df["patient_name"], df["admission_date"]))
]

quality_summary = pd.DataFrame({
    "column": df.columns,
    "null_count": [int(df[c].isna().sum()) for c in df.columns],
    "distinct_count": [int(df[c].nunique(dropna=True)) for c in df.columns],
})
quality_summary

In [ ]:
assert df["source_record_id"].is_unique, "Traceability IDs must be unique"
assert (df["discharge_date"] >= df["admission_date"]).fillna(False).all(), "Discharge must not precede admission"
assert (df["billing_amount"] >= 0).fillna(False).all(), "Billing amount must be non-negative"
print("Core source-data quality checks passed.")

## 5. Source-to-target mapping document

A Business Analyst should document what moves from the relational source to FHIR, how each field is transformed, and where assumptions remain. This mapping can be reviewed with developers, architects, data engineers, and stakeholders.

In [ ]:
mapping_path = PROJECT_DIR / "source_to_target_mapping.csv"
mapping_df = pd.read_csv(mapping_path)
mapping_df

### Key modeling decisions

- The dataset provides age but not date of birth, so the notebook does **not** invent a birth date. It stores recorded age in an extension.
- The medication column proves that a medication was recorded, but not that a prescription order exists. The notebook therefore uses `MedicationStatement`, not `MedicationRequest`.
- Free-text diagnosis and test values are mapped as text for the proof of concept. A production implementation should confirm terminology systems and code mappings with clinical and data-governance stakeholders.
- Patient identifiers are deterministic synthetic IDs generated from the source row for traceability.

## 6. FHIR R4 helper functions

In [ ]:
FHIR_RECORDED_AGE_EXTENSION = "https://example.org/fhir/StructureDefinition/recorded-age"

def slug(value: object, prefix: str) -> str:
    safe = re.sub(r"[^a-z0-9]+", "-", str(value).strip().lower()).strip("-")
    digest = hashlib.sha256(str(value).encode()).hexdigest()[:8]
    return f"{prefix}-{safe[:24]}-{digest}"

def patient_id(row) -> str:
    return f"patient-{row.source_record_id}"

def encounter_id(row) -> str:
    return f"encounter-{row.source_record_id}"

def normalize_gender(value: object) -> str:
    v = str(value).strip().lower()
    return {"male":"male","female":"female","other":"other"}.get(v, "unknown")

def iso_date(value) -> str | None:
    return None if pd.isna(value) else pd.Timestamp(value).date().isoformat()

def compact(resource: dict) -> dict:
    # Remove empty values while preserving valid false/zero values.
    if isinstance(resource, dict):
        return {k: compact(v) for k, v in resource.items() if v not in (None, "", [], {})}
    if isinstance(resource, list):
        return [compact(v) for v in resource if v not in (None, "", [], {})]
    return resource

## 7. Model FHIR resources from the relational data

In [ ]:
def make_patient(row) -> dict:
    return compact({
        "resourceType":"Patient",
        "id":patient_id(row),
        "identifier":[{"system":"urn:portfolio:synthetic-patient","value":row.source_record_id}],
        "name":[{"text":str(row.patient_name)}],
        "gender":normalize_gender(row.gender),
        "extension":[{"url":FHIR_RECORDED_AGE_EXTENSION,"valueInteger":int(row.age)}] if pd.notna(row.age) else [],
    })

def make_practitioner(doctor: str) -> dict:
    return {"resourceType":"Practitioner","id":slug(doctor,"practitioner"),"name":[{"text":str(doctor)}]}

def make_organization(hospital: str) -> dict:
    return {"resourceType":"Organization","id":slug(hospital,"organization"),"name":str(hospital)}

def make_encounter(row) -> dict:
    return compact({
        "resourceType":"Encounter",
        "id":encounter_id(row),
        "status":"finished",
        "class":{"system":"http://terminology.hl7.org/CodeSystem/v3-ActCode","code":"IMP","display":"inpatient encounter"},
        "type":[{"text":str(row.admission_type)}],
        "subject":{"reference":f"Patient/{patient_id(row)}"},
        "participant":[{"individual":{"reference":f"Practitioner/{slug(row.doctor,'practitioner')}","display":str(row.doctor)}}],
        "serviceProvider":{"reference":f"Organization/{slug(row.hospital,'organization')}","display":str(row.hospital)},
        "period":{"start":iso_date(row.admission_date),"end":iso_date(row.discharge_date)},
        "location":[{"location":{"display":f"Room {row.room_number}"}}],
    })

def make_condition(row) -> dict:
    return compact({
        "resourceType":"Condition",
        "id":f"condition-{row.source_record_id}",
        "clinicalStatus":{"coding":[{"system":"http://terminology.hl7.org/CodeSystem/condition-clinical","code":"active"}]},
        "subject":{"reference":f"Patient/{patient_id(row)}"},
        "encounter":{"reference":f"Encounter/{encounter_id(row)}"},
        "code":{"text":str(row.medical_condition)},
        "recordedDate":iso_date(row.admission_date),
    })

def make_blood_type_observation(row) -> dict:
    return compact({
        "resourceType":"Observation",
        "id":f"observation-blood-{row.source_record_id}",
        "status":"final",
        "code":{"text":"Blood type"},
        "subject":{"reference":f"Patient/{patient_id(row)}"},
        "encounter":{"reference":f"Encounter/{encounter_id(row)}"},
        "effectiveDateTime":iso_date(row.admission_date),
        "valueCodeableConcept":{"text":str(row.blood_type)},
    })

def make_test_result_observation(row) -> dict:
    return compact({
        "resourceType":"Observation",
        "id":f"observation-test-{row.source_record_id}",
        "status":"final",
        "code":{"text":"Admission test result"},
        "subject":{"reference":f"Patient/{patient_id(row)}"},
        "encounter":{"reference":f"Encounter/{encounter_id(row)}"},
        "effectiveDateTime":iso_date(row.admission_date),
        "valueCodeableConcept":{"text":str(row.test_results)},
    })

def make_medication_statement(row) -> dict:
    return compact({
        "resourceType":"MedicationStatement",
        "id":f"medicationstatement-{row.source_record_id}",
        "status":"active",
        "subject":{"reference":f"Patient/{patient_id(row)}"},
        "context":{"reference":f"Encounter/{encounter_id(row)}"},
        "dateAsserted":iso_date(row.admission_date),
        "medicationCodeableConcept":{"text":str(row.medication)},
    })

def make_coverage(row) -> dict:
    return compact({
        "resourceType":"Coverage",
        "id":f"coverage-{row.source_record_id}",
        "status":"active",
        "beneficiary":{"reference":f"Patient/{patient_id(row)}"},
        "payor":[{"display":str(row.insurance_provider)}],
    })

def make_claim(row) -> dict:
    return compact({
        "resourceType":"Claim",
        "id":f"claim-{row.source_record_id}",
        "status":"active",
        "type":{"coding":[{"system":"http://terminology.hl7.org/CodeSystem/claim-type","code":"institutional"}]},
        "use":"claim",
        "patient":{"reference":f"Patient/{patient_id(row)}"},
        "created":iso_date(row.discharge_date) or iso_date(row.admission_date),
        "provider":{"reference":f"Organization/{slug(row.hospital,'organization')}"},
        "priority":{"coding":[{"system":"http://terminology.hl7.org/CodeSystem/processpriority","code":"normal"}]},
        "total":{"value":float(row.billing_amount),"currency":"CAD"},
    })

In [ ]:
patients = [make_patient(r) for r in df.itertuples(index=False)]
encounters = [make_encounter(r) for r in df.itertuples(index=False)]
conditions = [make_condition(r) for r in df.itertuples(index=False)]
blood_observations = [make_blood_type_observation(r) for r in df.itertuples(index=False)]
test_observations = [make_test_result_observation(r) for r in df.itertuples(index=False)]
medication_statements = [make_medication_statement(r) for r in df.itertuples(index=False)]
coverages = [make_coverage(r) for r in df.itertuples(index=False)]
claims = [make_claim(r) for r in df.itertuples(index=False)]

practitioners = [make_practitioner(x) for x in sorted(df["doctor"].dropna().astype(str).unique())]
organizations = [make_organization(x) for x in sorted(df["hospital"].dropna().astype(str).unique())]

resource_counts = pd.DataFrame({
    "resource_type":["Patient","Encounter","Condition","Observation - Blood Type","Observation - Test Result",
                     "MedicationStatement","Coverage","Claim","Practitioner","Organization"],
    "count":[len(patients),len(encounters),len(conditions),len(blood_observations),len(test_observations),
             len(medication_statements),len(coverages),len(claims),len(practitioners),len(organizations)]
})
resource_counts

In [ ]:
print(json.dumps(patients[0], indent=2))

## 8. Create transaction bundles

A transaction Bundle groups related FHIR resources and declares RESTful request metadata for each entry.

In [ ]:
def entry(resource: dict) -> dict:
    return {
        "fullUrl":f"urn:uuid:{resource['id']}",
        "resource":resource,
        "request":{"method":"PUT","url":f"{resource['resourceType']}/{resource['id']}"},
    }

practitioner_lookup = {x["id"]:x for x in practitioners}
organization_lookup = {x["id"]:x for x in organizations}

def make_transaction_bundle(row) -> dict:
    resources = [
        make_patient(row),
        practitioner_lookup[slug(row.doctor,"practitioner")],
        organization_lookup[slug(row.hospital,"organization")],
        make_encounter(row),
        make_condition(row),
        make_blood_type_observation(row),
        make_test_result_observation(row),
        make_medication_statement(row),
        make_coverage(row),
        make_claim(row),
    ]
    return {"resourceType":"Bundle","type":"transaction","entry":[entry(r) for r in resources]}

bundles = [make_transaction_bundle(r) for r in df.itertuples(index=False)]
print(json.dumps(bundles[0], indent=2)[:6000])

## 9. Lightweight validation and JSON export

The checks below confirm required structure before API testing. Production validation should also use the target FHIR server's `$validate` operation and agreed implementation profiles.

In [ ]:
def validate_resource_shape(resource: dict) -> list[str]:
    issues = []
    if "resourceType" not in resource:
        issues.append("Missing resourceType")
    if "id" not in resource and resource.get("resourceType") != "Bundle":
        issues.append("Missing id")
    if resource.get("resourceType") == "Patient" and "identifier" not in resource:
        issues.append("Patient missing identifier")
    if resource.get("resourceType") == "Encounter" and "subject" not in resource:
        issues.append("Encounter missing subject")
    return issues

all_resources = (
    patients + practitioners + organizations + encounters + conditions +
    blood_observations + test_observations + medication_statements + coverages + claims
)
validation_results = [
    {"resourceType":r.get("resourceType"),"id":r.get("id"),"issues":validate_resource_shape(r)}
    for r in all_resources
]
validation_df = pd.DataFrame(validation_results)
validation_df["is_valid_shape"] = validation_df["issues"].apply(lambda x: len(x) == 0)
validation_df["is_valid_shape"].value_counts()

In [ ]:
def write_ndjson(filename: str, resources: list[dict]):
    path = OUTPUT_DIR / filename
    with path.open("w", encoding="utf-8") as f:
        for resource in resources:
            f.write(json.dumps(resource, separators=(",", ":")) + "\n")
    return path

exported = {
    "patients": write_ndjson("patients.ndjson", patients),
    "encounters": write_ndjson("encounters.ndjson", encounters),
    "conditions": write_ndjson("conditions.ndjson", conditions),
    "observations": write_ndjson("observations.ndjson", blood_observations + test_observations),
    "medication_statements": write_ndjson("medication_statements.ndjson", medication_statements),
    "coverages": write_ndjson("coverages.ndjson", coverages),
    "claims": write_ndjson("claims.ndjson", claims),
}
bundle_path = OUTPUT_DIR / "sample_transaction_bundle.json"
bundle_path.write_text(json.dumps(bundles[0], indent=2), encoding="utf-8")
exported["sample_bundle"] = bundle_path
exported

## 10. SQL analysis for data validation and source-to-target review

These queries demonstrate SQL-based analysis of the source data. Locally, SQLite provides a simple execution path. The Databricks section follows with Spark SQL.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
df.to_sql("healthcare_source", conn, index=False, if_exists="replace")

sql_query = '''
SELECT
    medical_condition,
    COUNT(*) AS encounter_count,
    ROUND(AVG(billing_amount), 2) AS avg_billing_amount,
    ROUND(AVG(julianday(discharge_date) - julianday(admission_date)), 2) AS avg_length_of_stay_days
FROM healthcare_source
GROUP BY medical_condition
ORDER BY encounter_count DESC, avg_billing_amount DESC;
'''
pd.read_sql_query(sql_query, conn)

In [ ]:
quality_query = '''
SELECT
    SUM(CASE WHEN patient_name IS NULL OR TRIM(patient_name) = '' THEN 1 ELSE 0 END) AS missing_patient_name,
    SUM(CASE WHEN admission_date IS NULL THEN 1 ELSE 0 END) AS missing_admission_date,
    SUM(CASE WHEN discharge_date < admission_date THEN 1 ELSE 0 END) AS invalid_date_sequence,
    SUM(CASE WHEN billing_amount < 0 THEN 1 ELSE 0 END) AS negative_billing_amount
FROM healthcare_source;
'''
pd.read_sql_query(quality_query, conn)

## 11. Databricks / Spark transformation pattern

Run this section inside Databricks or a local PySpark environment. Databricks notebooks can register temporary views, run Spark SQL, and write JSON outputs for downstream processing.

In [ ]:
# Databricks / PySpark example
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.getOrCreate()
#
# spark_df = spark.createDataFrame(df.astype(object).where(pd.notna(df), None))
# spark_df.createOrReplaceTempView("healthcare_source")
#
# display(spark.sql('''
# SELECT
#   medical_condition,
#   COUNT(*) AS encounter_count,
#   ROUND(AVG(billing_amount), 2) AS avg_billing_amount
# FROM healthcare_source
# GROUP BY medical_condition
# ORDER BY encounter_count DESC
# '''))

In [ ]:
# Databricks JSON export example
# spark.read.json(str(OUTPUT_DIR / "patients.ndjson")).createOrReplaceTempView("fhir_patients")
# spark.sql("SELECT resourceType, COUNT(*) AS resource_count FROM fhir_patients GROUP BY resourceType").show()
#
# Optional DBFS / cloud storage export:
# spark.read.json(str(OUTPUT_DIR / "patients.ndjson")).write.mode("overwrite").json("/tmp/ops-fhir-demo/patients")

### Databricks SQL cell example

Copy the following into a Databricks `%sql` cell after registering `healthcare_source`:

```sql
SELECT
  admission_type,
  medical_condition,
  COUNT(*) AS encounter_count,
  ROUND(AVG(billing_amount), 2) AS avg_billing_amount
FROM healthcare_source
GROUP BY admission_type, medical_condition
ORDER BY encounter_count DESC;
```

## 12. Optional FHIR REST API validation

The code is disabled by default to prevent accidental writes. Review the payload and use synthetic data only. The included Postman collection provides a reusable API-testing workflow.

In [ ]:
import requests

FHIR_BASE_URL = "https://hapi.fhir.org/baseR4"
EXECUTE_API_CALLS = False

if EXECUTE_API_CALLS:
    metadata = requests.get(
        f"{FHIR_BASE_URL}/metadata",
        headers={"Accept":"application/fhir+json"},
        timeout=30,
    )
    metadata.raise_for_status()
    print("Capability resource:", metadata.json().get("resourceType"))

    validation = requests.post(
        f"{FHIR_BASE_URL}/Patient/$validate",
        headers={"Content-Type":"application/fhir+json","Accept":"application/fhir+json"},
        json=patients[0],
        timeout=30,
    )
    print("Validation status:", validation.status_code)
    print(json.dumps(validation.json(), indent=2)[:4000])

    # Review before enabling transaction write:
    # response = requests.post(
    #     FHIR_BASE_URL,
    #     headers={"Content-Type":"application/fhir+json","Accept":"application/fhir+json"},
    #     json=bundles[0],
    #     timeout=30,
    # )
    # print(response.status_code, response.text[:4000])
else:
    print("API calls are disabled. Import the Postman collection or set EXECUTE_API_CALLS=True after review.")

## 13. Postman API-testing workflow

Import `postman_fhir_r4_validation_collection.json` into Postman. It includes:

1. `GET /metadata` to retrieve the server CapabilityStatement.
2. `POST /Patient/$validate` to validate a synthetic Patient JSON payload.
3. `POST /` to submit a synthetic transaction Bundle.
4. `GET /Patient?_count=5` to test FHIR search.

The collection contains response assertions for status codes, JSON output, and returned `resourceType`.

## 14. Azure DevOps-ready user stories and traceability

The included `azure_devops_backlog.csv` is structured for backlog import or manual entry. It contains a Feature and User Stories with descriptions, acceptance criteria, priorities, and tags.

In [ ]:
backlog_df = pd.read_csv(PROJECT_DIR / "azure_devops_backlog.csv")
backlog_df

## 15. Requirements-validation checklist for stakeholder review

Use this during a mock JAD session or technical review:

| Review question | Stakeholder |
|---|---|
| Which FHIR version and implementation guide should the target solution follow? | Architect / Product Owner |
| Are terminology systems required for diagnosis, medication, and test-result fields? | Clinical Data Specialist |
| Should age-only records be rejected, enriched, or retained using an approved profile? | Data Governance / Business Owner |
| Is `MedicationStatement` appropriate, or does the source contain enough information for `MedicationRequest`? | Clinical SME / Architect |
| Which API interactions are allowed: read, create, update, search, batch, transaction, validation? | API Team |
| Which Databricks workspace, catalog, schema, storage location, and lineage controls apply? | Data Engineer / Platform Team |
| What acceptance criteria and test evidence are required before release? | QA / Product Owner |

## 16. Resume-safe project wording after completing the notebook

**Healthcare Data Interoperability Proof of Concept | FHIR R4, SQL, JSON, Postman, Databricks, Azure DevOps**

- Mapped synthetic relational hospital records to FHIR R4 resources, including Patient, Encounter, Condition, Observation, MedicationStatement, Coverage, and Claim.
- Created source-to-target mapping documentation and generated JSON transaction bundles for API-based interoperability testing.
- Built SQL and Spark transformation examples for a Databricks workflow and created an Agile backlog with user stories, acceptance criteria, and traceability artifacts.
- Validated synthetic FHIR payloads using an importable Postman collection and REST API test workflow.